In [1]:
import joblib
import pandas as pd
import ipywidgets as widgets

# Load the encoder, scaler, and best_model from saved files
encoder = joblib.load("encoder.pkl")
scaler = joblib.load("scaler.pkl")
best_model = joblib.load("best_model.pkl")

#MODEL DEPLOYMENT

social_hours_widget = widgets.FloatSlider(
    value=5.0,
    min=0.0,
    max=14.0,
    step=1.0,
    description='Social Media hours',
    style={'description_width': 'initial'},
    continuous_update = False
)
sleep_hours_widget = widgets.FloatSlider(
    value=6.0,
    min=2.0,
    max=11.15,
    step=1.0,
    description='Sleep Hours',
    style={'description_width': 'initial'},
    continuous_update = False
)
mental_health_widget = widgets.FloatSlider(
    value=60.0,
    min=32.56,
    max=91.76,
    step=0.1,
    description='Mental Health Score:',
    style={'description_width': 'initial'},
    continuous_update = False
)
burnout_widget = widgets.Dropdown(
    options=["Low","Moderate","High", "Severe"],
    value="Moderate",
    description="Burnout Level:",
    style={"description_width": "initial"},
    continuous_update = False
)
Academic_performance__widget = widgets.FloatSlider(
    value=60.0,
    min=23.99,
    max=99.98,
    step=0.01,
    description='Academic Performance Score',
    style={'description_width': 'initial'},
    continuous_update = False
)
predict_button = widgets.Button(description="Predict Result",button_style="success",icon = "check")
output = widgets.Output()
from IPython.display import clear_output, display

def predict_student_risk(button):
    with output:
        clear_output()

        student_input = pd.DataFrame({
            "Daily_Social_Media_Hours": [
                social_hours_widget.value
            ],
            "Sleep_Hours": [
                sleep_hours_widget.value
            ],
            "Mental_Health_Score":[
                mental_health_widget.value
            ],
            "Burnout_Level":[
                burnout_widget.value
            ],
            "Academic_Performance_Score":[
                Academic_performance__widget.value]})
        cat_cols_new_student = student_input.select_dtypes(include=['object','category']).columns
        student_input[cat_cols_new_student] = encoder.transform(student_input[cat_cols_new_student])

        student_input_scaled = scaler.transform(student_input)

        predicted_class = best_model.predict(
            student_input_scaled
        )[0]

        predicted_probabilities = (
            best_model.predict_proba(
                student_input_scaled
            )[0])
        print("=" * 45)
        print("STUDENT RESULT PREDICTION")
        print("=" * 45)

        display(student_input)

        if predicted_class == 0:
            print("✅ Prediction: No Academic Failure Risk")
        else:
            print("❌ Prediction: Academic Failure Risk")

        print("\nPrediction Probabilities:")

        for class_name, probability in zip(
            best_model.classes_,
            predicted_probabilities
        ):
            print(
                f"{class_name}: "
                f"{probability * 100:.2f}%"
            )


predict_button.on_click(
    predict_student_risk
)

application = widgets.VBox([
    widgets.HTML(
        value="<h3>🎓 Student Risk/No Risk Predictor</h3>"
    ),
    widgets.HTML(
        value=(
            "<p>Enter the student's information "
            "and click Predict Result.</p>"
        )
    ),
    social_hours_widget
    ,sleep_hours_widget
    ,mental_health_widget
    ,burnout_widget
    ,Academic_performance__widget
    ,predict_button,
    output
])
display(application)